In [1]:
import pandas as pd

df = pd.read_csv("vannverk_aar.csv", encoding="utf-8-sig")

print(df.shape)                                   # forventet: ca. 30 000 rader, ca. 75 kolonner
print(df.duplicated(["mtid_vf", "periode"]).sum())  # skal være 0
print(df.periode.value_counts().sort_index())     # rader per år, 2008 til 2025
print(df.isna().mean().sort_values(ascending=False).head(20))  # hvor mye som mangler
print(df[["total_analyser", "total_avvik"]].describe())

(36034, 80)
0
periode
2008     412
2009    2004
2010    2298
2011    2287
2012    2277
2013    2315
2014    2321
2015    2142
2016    2262
2017    2178
2018    2022
2019    1998
2020    2017
2021    1981
2022    1879
2023    1899
2024    1903
2025    1839
Name: count, dtype: int64
noedvann_inngaar_alt_kilde    0.624688
max_vann_dogn                 0.517900
max_vann_pers                 0.506244
forbr_annet                   0.472748
forbr_industri                0.453877
forbr_tjytende                0.453044
forbr_primnaering             0.446745
forbr_fritidsboliger          0.441416
forbr_lekkasje                0.422101
forbr_fast_bosetting          0.406200
ledn_gjsn                     0.211828
ledn_avvik                    0.211328
ledn_analyser                 0.195454
vann_mottatt                  0.177555
enterok_gjsn                  0.153438
enterok_avvik                 0.149248
enterok_analyser              0.136815
ionebytte                     0.124521
ozon_desinf     

In [2]:
d = df[df.total_analyser > 0].copy()
print(len(d), (d.total_avvik > 0).mean())   # andel vannverk-år med minst ett avvik

cols = ["vannprod", "ant_fastboende", "forbr_fast_bosetting",
        "max_vann_dogn", "noedvann_inngaar_alt_kilde"]
print(df.groupby("periode")[cols].apply(lambda x: x.isna().mean()).round(2))

print(df.nlargest(5, "total_analyser")[["navn", "periode", "total_analyser", "total_avvik"]])

34307 0.5662692744920862
         vannprod  ant_fastboende  forbr_fast_bosetting  max_vann_dogn  \
periode                                                                  
2008          0.0            0.01                  0.98           1.00   
2009          0.0            0.01                  0.53           1.00   
2010          0.0            0.00                  0.52           1.00   
2011          0.0            0.00                  0.50           1.00   
2012          0.0            0.00                  0.48           1.00   
2013          0.0            0.00                  0.47           0.71   
2014          0.0            0.00                  0.48           0.68   
2015          0.0            0.00                  0.38           0.57   
2016          0.0            0.00                  0.39           0.48   
2017          0.0            0.00                  0.37           0.38   
2018          0.0            0.00                  0.35           0.33   
2019         

In [3]:
d = df[(df.total_analyser > 0) & df.total_avvik.notna() & (df.periode >= 2009)].copy()
d["avvik"] = (d.total_avvik > 0).astype(int)
print(len(d), d.avvik.mean())

33752 0.57531405546338


In [6]:
oversikt = pd.DataFrame({
    "type": df.dtypes,
    "andel_mangler": df.isna().mean().round(2),
    "antall_unike": df.nunique(),
    "vanligste_verdi_andel": df.apply(lambda s: s.value_counts(normalize=True, dropna=True).iloc[0] if s.notna().any() else None).round(2),
})
print(oversikt.sort_values("andel_mangler").to_string())

                               type  andel_mangler  antall_unike  vanligste_verdi_andel
mtid_vf                      object           0.00          4053                   0.00
ant_hytter                  float64           0.00           299                   0.32
ant_husstander              float64           0.00           696                   0.14
ant_personer_max            float64           0.00           736                   0.05
ant_fastboende              float64           0.00           783                   0.16
beredsk_oppd                 object           0.00             2                   0.54
beredsk_ovelse               object           0.00             2                   0.68
vannuttak                   float64           0.00         18608                   0.06
aktiv                        object           0.00             2                   0.76
navn                         object           0.00          3423                   0.00
periode                       in

In [8]:
c = pd.read_csv("vannverk_clean.csv", encoding="utf-8-sig")

rens = ["antall_anlegg", "uv", "klorering", "koagulering",
        "membranfiltrering", "siling", "lufting", "ph_justering"]

print(c[rens].isna().mean().round(3))                       # andel tomme per kolonne
print(c[rens].isna().all(axis=1).mean().round(3))           # andel rader der ALLE er tomme
print(c[c.uv.isna()].groupby("orgform").size().sort_values(ascending=False).head(5))

antall_anlegg        0.119
uv                   0.119
klorering            0.119
koagulering          0.119
membranfiltrering    0.119
siling               0.119
lufting              0.119
ph_justering         0.119
dtype: float64
0.119
orgform
KOMM    1380
SA       663
AS       430
FLI      406
BEDR     370
dtype: int64
